[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h02_fermat.ipynb)

# Hunt for cubes: Fermat's Last Theorem

Nearly four hundred years ago, Pierre de Fermat wrote in the margin of a book that aⁿ + bⁿ = cⁿ has no solutions
in whole numbers when n is bigger than 2, and that the margin was too small for his proof. Squares split into two
squares all the time (9 + 16 = 25). Cubes never do. Here you can hunt for cubes yourself, catch a near miss that
fools a calculator, and watch Fermat's own method, infinite descent, at work.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
from decimal import Decimal, getcontext
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, IntText, ToggleButton

getcontext().prec = 60
SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

def digit_sum(n):
    return sum(int(d) for d in str(n))

print("Ready. Run the next cell.")

## 1 · Squares split, again and again

A right triangle's sides a, b, c with a² + b² = c²: 3, 4, 5 is the famous one. Drag the slider to raise the
longest side allowed and count them. They never run out.

In [ ]:
def triples(longest=100):
    found = [(a, b, c) for c in range(1, longest + 1) for a in range(1, c) for b in range(a, c) if a * a + b * b == c * c]
    print(f"{len(found)} right triangles with whole sides and the long side up to {longest}")
    print("the first few:", ", ".join(f"{a}-{b}-{c}" for a, b, c in found[:8]))
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter([a for a, b, c in found], [b for a, b, c in found], color=TEAL, s=18)
    ax.scatter([3], [4], color=CORAL, s=60, label="3, 4, 5")
    ax.set_xlabel("a"); ax.set_ylabel("b"); ax.set_title("every dot: a² + b² = c² exactly"); ax.legend(); plt.show()

interact(triples, longest=IntSlider(value=100, min=5, max=300, step=5, description="longest side"));

## 2 · Hunt for cubes

Try every pair of cubes up to a limit, add them, and ask: is the total a cube? Drag the limit up. You'll find no
exact hit, ever, but you'll find near misses: totals one away from a cube. The first is 9³ + 10³ = 1,729, and
12³ = 1,728.

In [ ]:
def cube_hunt(limit=100):
    cubes = {c ** 3: c for c in range(1, int(2 ** (1 / 3) * limit) + 3)}
    exact, near = 0, []
    for x in range(1, limit + 1):
        for y in range(x, limit + 1):
            s = x ** 3 + y ** 3
            if s in cubes: exact += 1
            for d in (-1, 1):
                if x > 1 and s + d in cubes: near.append((x, y, cubes[s + d], d))   # 1³ + y³ always misses by 1
    print(f"pairs tried: {limit * (limit + 1) // 2:,}    exact hits: {exact}")
    print("near misses (one away from a cube; leaving out 1³ + y³):")
    for x, y, z, d in sorted(near, key=lambda m: m[2])[:12]:
        print(f"   {x}³ + {y}³ = {x ** 3 + y ** 3:,}    {z}³ = {z ** 3:,}    missed by {abs(d)}")

interact(cube_hunt, limit=IntSlider(value=100, min=10, max=1000, step=10, description="up to"));

## 3 · Homer's equation

In a 1998 episode of The Simpsons, Homer writes 3987¹² + 4365¹² = 4472¹². A ten-digit calculator agrees. Type in
any equation of this kind. The first line is what a calculator shows; the rest is the truth, and the digit test
you can do in your head.

In [ ]:
def homer(a=3987, b=4365, c=4472, n=12):
    left, right = a ** n + b ** n, c ** n
    root = Decimal(left) ** (Decimal(1) / n)
    print(f"a calculator shows:  ({a}^{n} + {b}^{n})^(1/{n}) = {root:.6f}")
    print(f"more digits:         {root:.15f}")
    print(f"exactly equal?       {left == right}")
    print(f"\ndigit sums: {a} → {digit_sum(a)},  {b} → {digit_sum(b)},  {c} → {digit_sum(c)}")
    for p in (2, 3):
        l, r = left % p, right % p
        if l != r:
            print(f"left side leaves {l} when divided by {p}; right side leaves {r}: they can't be equal")
            break

interact(homer, a=IntText(value=3987, description="a"), b=IntText(value=4365, description="b"),
         c=IntText(value=4472, description="c"), n=IntText(value=12, description="power"));

## 4 · Is a square ever exactly twice a square?

7² = 49 and 2 × 5² = 50. 17² = 289 and 2 × 12² = 288. The near misses go on forever. Drag the slider to search
further: there is never an exact hit. Fermat's infinite descent says why: any exact hit a² = 2b² would give a
smaller one, b² = 2(a/2)², and whole numbers can't go down forever.

In [ ]:
def twice(up_to=1000):
    near, exact = [], []
    for b in range(1, up_to + 1):
        a = math.isqrt(2 * b * b)
        for aa in (a, a + 1):
            if aa * aa == 2 * b * b: exact.append((aa, b))
            elif abs(aa * aa - 2 * b * b) == 1: near.append((aa, b))
    print(f"b from 1 to {up_to:,}:   exact hits: {len(exact)}")
    for a, b in near:
        print(f"   {a}² = {a * a:,}    2 × {b}² = {2 * b * b:,}")

interact(twice, up_to=IntSlider(value=1000, min=10, max=100000, step=10, description="b up to"));

**The staircase.** Suppose a² = 2b². Then a² is even, so a is even: a = 2c. Then 4c² = 2b², so b² = 2c². Now (b, c)
is a new solution, made of smaller numbers. Do it again and again: the numbers keep shrinking, but whole numbers
can't shrink forever. So there was never a first solution.

## 5 · An elliptic curve on a clock

The curve y² = x³ − x helped settle Fermat's question, and curves like it now protect messages on your phone. On a
phone they live on a "clock": arithmetic that wraps round at a prime p. Drag p and watch the points.

In [ ]:
def clock_curve(p=23):
    if p < 3 or any(p % d == 0 for d in range(2, int(p ** 0.5) + 1)):
        print("choose a prime"); return
    pts = [(x, y) for x in range(p) for y in range(p) if (y * y - (x ** 3 - x)) % p == 0]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter([x for x, y in pts], [y for x, y in pts], color=LILAC, s=max(8, 2400 // p))
    ax.set_xlim(-1, p); ax.set_ylim(-1, p); ax.set_aspect("equal")
    ax.set_title(f"y² = x³ − x on a clock of {p}: {len(pts)} points (+1 at infinity)"); plt.show()

interact(clock_curve, p=IntSlider(value=23, min=3, max=211, step=2, description="prime p"));

## 6 · The challenge from the video

A 1995 episode of The Simpsons shows 1782¹² + 1841¹² = 1922¹². A calculator agrees again. Can you show it's false
without one? Think first, then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("1782¹² + 1841¹² = 1922¹²  ?   (press the button when you're ready)"); return
    print("1782 is even, 1841 is odd: even¹² + odd¹² = even + odd = odd.")
    print("1922 is even, so 1922¹² is even. An odd number can't equal an even one.")
    print(f"(the calculator's 1922.000000 is really {Decimal(1782 ** 12 + 1841 ** 12) ** (Decimal(1) / 12):.10f})")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
MacTutor History of Mathematics: Fermat's Last Theorem; Pierre de Fermat; Sophie Germain; Andrew Wiles. NOVA, *The
Proof* (PBS, 1997), transcript. A. Wiles, *Modular elliptic curves and Fermat's Last Theorem*, Annals of Mathematics
141 (1995). Simon Singh on the Simpsons equations. All retrieved 2026-10-27.